# Week 4: Make Our Disease Simulation Interactive
**7 to 13 November 2026**

**Goal:** Add new controls to the town we made in Week 2. Use the `run_simulation(inputs)` structure from Week 3.

## Role 1: Jayden | Check the disease inputs

1. Use `select_disease(name, scenarios)` to look up a disease in the Week 3 `scenarios`. Show an error if the name is unknown.
2. Check the four inputs listed in `REQUIRED`. Return the missing names if any value is `None`. Otherwise, return a copy of the inputs.
3. Show three practice tests with complete inputs, a missing value and an unknown name. Keep the Week 3 inputs as they are.


In [ ]:
REQUIRED = ("infection_chance", "exposed_days", "infectious_days", "meetings_per_day")

def select_disease(name, scenarios):
    # Step 1: check that name is in scenarios
    if name not in scenarios: 
        return 'Error, the disease name is not in scenarios.'
    
    # Step 2: get the selected dictionary
    parameters = scenarios[name]

    # Step 3: collect missing parameter names
    missing = []

    for parameter in REQUIRED:
        if parameters[parameter] is None: 
            missing.append(parameter)

    # Step 4: return (parameters, []) or (None, missing)
    if missing:
        return (None, missing)
    else:
        parameters_copy = parameters.copy()
        return (parameters_copy, [])

# TEST ONLY: make a small dictionary and test three cases.
practice_scenarios = {
    'practice_measles': {
        'infection_chance': 0.3,
        'exposed_days': 2,
        'infectious_days': 4,
        'meetings_per_day':3
    },
    'practice_influenza':{
        'infection_chance': None,
        'exposed_days': 4,
        'infectious_days': None,
        'meetings_per_day': 3
    },
    'practice_covid_19':{
        'infection_chance': .6,
        'exposed_days': 2,
        'infectious_days': 5,
        'meetings_per_day': 3
    }

}

# TEST for no None values
print(select_disease('practice_measles', practice_scenarios))

# TEST for None values
print(select_disease('practice_influenza', practice_scenarios))

# TEST for disease name not in scenarious
print(select_disease('practice_chickenpox', practice_scenarios))

# Do not change the actual Week 3 scenarios.

({'infection_chance': 0.3, 'exposed_days': 2, 'infectious_days': 4, 'meetings_per_day': 3}, [])
(None, ['infection_chance', 'infectious_days'])
Error, the disease name is not in scenarios.


## Role 2: Alex | Vaccination control

1. In `apply_vaccination(town, percent, rng)`, accept percentages from 0 to 100.
2. List the susceptible residents. Use `round(len(eligible) * percent / 100)` and `rng.sample()` to pick that many IDs.
3. Set `protected` to `True` for those residents and `False` for the rest. Return the town and the set of selected IDs.
4. Skip protected residents in the infection check. Keep their health state as `S`. This exercise assumes full protection in the model.
5. Test 0%, 50% and 100%. Print how many were selected and confirm the first infectious resident was never selected.


In [ ]:
import random

def apply_vaccination(town, percent, rng):
    # Check percent is 0 to 100.
    # Find people whose state is "S".
    # Choose IDs with rng.sample().
    # Add person["protected"] = True/False.
    # Return town, selected_ids.
    pass

# Inside the existing infection condition add:
# and not other.get("protected", False)
# Test separately on a fresh town each time, using random.Random(42).

## Role 3: Linh | Stay-home control

1. In `choose_stay_home(infectious_ids, percent, rng)`, accept percentages from 0 to 100 and remove repeated IDs.
2. Use `round(len(ids) * percent / 100)` and `rng.sample()` to choose who stays home. Return their IDs as a set.
3. Call the function once each day. Skip meetings started by these residents and keep their recovery timers running.
4. Show tests for 0%, 50% and 100% using the same IDs and seed. At 100%, these residents should start no meetings.


In [ ]:
def choose_stay_home(infectious_ids, percent, rng):
    # Check percent.
    # Create a unique list of IDs.
    # Calculate the number who stay home.
    # Select with rng.sample() and return a set.
    pass

# Each simulated day:
# infectious_ids = [p["id"] for p in town if p["state"] == "I"]
# staying_home = choose_stay_home(infectious_ids, stay_home_percent, rng)
# In the current meeting loop:
# if person["id"] in staying_home: continue

## Role 4: Aayush | Connect the controls

1. Use `ipywidgets` to make a disease dropdown, sliders from 0 to 100 for vaccination and staying home, and an output area.
2. In `update_output()`, read the controls and use Jayden’s `select_disease()`. Show a message if inputs are missing. If they are ready, call the shared runner with both percentages and pass the result to Ei Kay’s `show_results()`.
3. Use `.observe()` on all three controls. Display them and call `update_output()` once to show the starting result.
4. Try practice inputs and show that moving a slider updates the output.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

def make_controls(scenarios, run_with_controls, show_results):
    disease = widgets.Dropdown(options=["measles", "influenza", "covid_19"], description="Disease")
    vaccination = widgets.IntSlider(value=0, min=0, max=100, step=10, description="Vaccinated")
    stay_home = widgets.IntSlider(value=0, min=0, max=100, step=10, description="Stay home")
    output = widgets.Output()

    def update_output(change=None):
        with output:
            output.clear_output(wait=True)
            # Get parameters using select_disease.
            # Show missing values, or call run_with_controls(...).
            # Call show_results(result).
            pass

    # Add .observe() for each widget.
    # display(disease, vaccination, stay_home, output)
    # update_output()

## Role 5: Ei Kay | Check and show results

1. In `validate_result()`, check that counts, snapshots and meetings cover days 0–60. Each day should have 100 people in total and no negative counts.
2. In `show_results()`, check the results first. Plot S, E, I and R, and use Week 2’s `draw_town()` to display the town on the selected day.
3. In `make_result_row()`, save the disease, percentages, seed, highest infectious count and number ever infected. Count people who were infected at any point in the run.
4. Test one valid and one broken result. Submit the graphs, validation code and a sample CSV for Week 5.


In [19]:
import matplotlib.pyplot as plt
import pandas as pd

def validate_result(result):
    problems = []
    expected = 61  # Day 0 to Day 60

    # Check list lengths.
    for key in ["days", "S", "E", "I", "R"]:
        if key not in result:
            problems.append(f"Missing key: {key}")
        elif len(result[key]) != expected:
            problems.append(f"{key} has {len(result[key])} values, expected {expected}")

    # Check population totals and non-negative values.
    if not problems:
        for d in range(expected):
            counts = [result[s][d] for s in ["S", "E", "I", "R"]]
            if any(c < 0 for c in counts):
                problems.append(f"Day {d}: negative count {counts}")
            if sum(counts) != 100:
                problems.append(f"Day {d}: S+E+I+R = {sum(counts)}, expected 100")

    # Check snapshots and meetings length.
    for key in ["snapshots", "meetings"]:
        if key not in result:
            problems.append(f"Missing key: {key}")
        elif len(result[key]) != expected:
            problems.append(f"{key} has {len(result[key])} entries, expected {expected}")

    # Return True if all checks pass.
    if problems:
        print("Validation FAILED:")
        for p in problems:
            print(" -", p)
        return False
    print("Validation passed.")
    return True

def show_results(result, day=30):
    # Call validate_result(result).
    if not validate_result(result):
        print("Graphs not drawn because the result is invalid.")
        return

    fig, (ax_line, ax_town) = plt.subplots(1, 2, figsize=(14, 6))

    # Draw S, E, I, R curves.
    labels = {"S": "Susceptible", "E": "Exposed", "I": "Infectious", "R": "Recovered"}
    for s in ["S", "E", "I", "R"]:
        ax_line.plot(result["days"], result[s], label=labels[s], color=COLORS[s])
    ax_line.axvline(day, color="gray", linestyle="--", linewidth=1)
    ax_line.set_title("SEIR Counts Over 60 Days")
    ax_line.set_xlabel("Day")
    ax_line.set_ylabel("Number of people")
    ax_line.legend()

    # Reuse Week 2 draw_town(...) for the selected day.
    states = get_states(result["snapshots"][day])
    draw_town(ax_town, states, result["meetings"][day], day)

    plt.tight_layout()
    plt.show()

def make_result_row(disease, vaccinated_percent, stay_home_percent, seed, result):
    # Get peak infectious count.
    peak_infectious = max(result["I"])

    # Compute total ever infected using the model's infection history.
    # Anyone who was E or I on any day was infected at some point.
    # Vaccinated people stay "S", so they are never counted.
    ever_infected = set()
    for snapshot in result["snapshots"]:
        for pid, state in enumerate(get_states(snapshot)):
            if state in ("E", "I"):
                ever_infected.add(pid)

    # Return a dictionary with one field per CSV column.
    return {
        "disease": disease,
        "vaccinated_percent": vaccinated_percent,
        "stay_home_percent": stay_home_percent,
        "seed": seed,
        "peak_infectious": peak_infectious,
        "total_ever_infected": len(ever_infected),
    }


# ----- Helpers used above -----

# Given in Week 2: grid spot and color for each person
POSITIONS = [(i % 10, i // 10) for i in range(100)]
COLORS = {"S": "#4C9BE0", "E": "#F0A030", "I": "#E04040", "R": "#50B060"}

# Week 2 draw_town(), copied unchanged so this notebook can use it
def draw_town(ax, states, meetings, day):
    ax.clear()
    ax.set_title(f"Virtual Town - Day {day}")
    for a, b in meetings:
        ax.plot([POSITIONS[a][0], POSITIONS[b][0]], [POSITIONS[a][1], POSITIONS[b][1]], color="gray", linewidth=0.5)
    for state, color in COLORS.items():
        xs = [POSITIONS[i][0] for i, s in enumerate(states) if s == state]
        ys = [POSITIONS[i][1] for i, s in enumerate(states) if s == state]
        ax.scatter(xs, ys, c=color, label=state)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend()

def get_states(snapshot):
    # A snapshot can be a list of letters ("S") or a list of person dictionaries.
    # draw_town() needs letters, so convert dictionaries to their "state".
    return [p["state"] if isinstance(p, dict) else p for p in snapshot]

def save_results_csv(rows, path):
    # Put result rows into a Pandas DataFrame and save it as CSV.
    df = pd.DataFrame(rows)
    df.to_csv(path, index=False)
    return df

## Role 6: Project leader | Review the work

Connect the five coding parts and check the tests before merging. Verify the research inputs separately. Explain the controls and results for the presentation.
